In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [3]:
import wandb

wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
from typing import Optional
import wandb


class TrainMonitor:

    def __init__(
        self,
        project: str,
        model_name: str,
        version: str = "v1",
        experiment: str = "baseline",
        mode: str = "online",
        config: Optional[dict] = None,
        tags: Optional[list] = None,
        notes: str = "",
        finish_previous: bool = True,
    ):

        self.project = project
        self.model_name = model_name
        self.version = version
        self.experiment = experiment

        self.run = wandb.init(
            project=project,
            name=f"{model_name}_{version}_{experiment}",
            config=config,
            tags=tags,
            notes=notes,
            mode=mode,
            reinit=finish_previous,
        )

    def monitor(self, metrics: dict, step: Optional[int] = None):
        """
        Log any metrics
        """
        wandb.log(metrics, step=step)

    def update_config(self, params: dict):
        wandb.config.update(params, allow_val_change=True)

    def watch(self, model):
        wandb.watch(model)

    def finish(self):
        wandb.finish()

In [5]:
monitor = TrainMonitor(
    project="24f1000781-t22026",
    model_name="Pretrained_backbone_finetune",
    version="v3.1",
    experiment="distilroberta_less_epochs_and_lr",
    config={
        "lr":6e-6,
        "batch_size":4,
        "epochs":10,
    }
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_144523-ujvrjyik
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Pretrained_backbone_finetune_v3.1_distilroberta_less_epochs_and_lr
wandb: ⭐️ View project at https://wandb.ai/iitm-swastik-indian-institute-of-technology-madras/24f1000781-t22026
wandb: 🚀 View run at https://wandb.ai/iitm-swastik-indian-institute-of-technology-madras/24f1000781-t22026/runs/ujvrjyik


In [6]:
# import pandas as pd
# import numpy as np
# import torch
# from dataclasses import dataclass
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# from transformers import (
#     AutoTokenizer,
#     AutoModelForMultipleChoice,
#     TrainingArguments,
#     Trainer,
#     PreTrainedTokenizerBase,
#     TrainerCallback
# )
# from transformers.utils import PaddingStrategy
# from datasets import Dataset
# from typing import Optional, Union, List

# class MonitorCallback(TrainerCallback):
#     def on_log(self, args, state, control, logs=None, **kwargs):
#         if logs is None or "eval_loss" not in logs:
#             return
        
#         try:
#             train_loss = 0.0
#             train_accuracy = 0.0
#             for past_log in reversed(state.log_history):
#                 if "loss" in past_log:
#                     train_loss = past_log["loss"]
#                     train_accuracy = past_log.get("train_accuracy", 0.0)
#                     break
                    
#             monitor.monitor({
#                 "epoch": round(state.epoch, 2) if state.epoch else 0,
#                 "train/loss": train_loss,
#                 "train/accuracy": train_accuracy,
#                 "validation/loss": logs.get("eval_loss", 0.0),
#                 "validation/accuracy": logs.get("eval_accuracy", 0.0),
#                 "validation/map_at_3": logs.get("eval_map@3", 0.0)
#             })
#         except NameError:
#             pass

# class CustomTrainer(Trainer):
#     """
#     Subclass Trainer to calculate training accuracy on the fly during the forward pass.
#     This avoids having to run a separate, slow evaluation loop over the training data.
#     """
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.train_correct = 0
#         self.train_total = 0

#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         labels = inputs.get("labels")
#         outputs = model(**inputs)
#         loss = outputs.get("loss")
#         logits = outputs.get("logits")

#         with torch.no_grad():
#             if logits is not None and labels is not None:
#                 preds = torch.argmax(logits, dim=1)
#                 self.train_correct += (preds == labels).sum().item()
#                 self.train_total += labels.size(0)

#         return (loss, outputs) if return_outputs else loss

#     def log(self, logs: dict, *args, **kwargs):
#         if "loss" in logs:
#             if self.train_total > 0:
#                 logs["train_accuracy"] = self.train_correct / self.train_total
#             else:
#                 logs["train_accuracy"] = 0.0
#             self.train_correct = 0
#             self.train_total = 0
#         super().log(logs)


# class TFIDFRetriever:
#     def __init__(self, corpus: List[str], max_features: int = 50000):
#         self.corpus = np.array(corpus)
#         self.vectorizer = TfidfVectorizer(
#             stop_words='english', 
#             ngram_range=(1, 2), 
#             max_features=max_features,
#             dtype=np.float32
#         )
#         self.tfidf_matrix = self.vectorizer.fit_transform(self.corpus)

#     def retrieve(self, queries: List[str], top_k: int = 3) -> List[str]:
#         query_vectors = self.vectorizer.transform(queries)
#         similarities = cosine_similarity(query_vectors, self.tfidf_matrix)
        
#         retrieved_contexts = []
#         for sim in similarities:
#             top_indices = sim.argsort()[-top_k:][::-1]
#             context_chunks = self.corpus[top_indices]
#             retrieved_contexts.append(" ".join(context_chunks))
        
#         return retrieved_contexts

# @dataclass
# class DataCollatorForMultipleChoice:
#     tokenizer: PreTrainedTokenizerBase
#     padding: Union[bool, str, PaddingStrategy] = True
#     max_length: Optional[int] = None
#     pad_to_multiple_of: Optional[int] = None

#     def __call__(self, features):
#         label_name = "label" if "label" in features[0].keys() else "labels"
#         labels = [feature.pop(label_name) for feature in features]
#         batch_size = len(features)
#         num_choices = len(features[0]["input_ids"])
        
#         flattened_features = [
#             [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
#         ]
#         flattened_features = sum(flattened_features, [])
        
#         batch = self.tokenizer.pad(
#             flattened_features,
#             padding=self.padding,
#             max_length=self.max_length,
#             pad_to_multiple_of=self.pad_to_multiple_of,
#             return_tensors="pt",
#         )
        
#         batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
#         batch["labels"] = torch.tensor(labels, dtype=torch.int64)
#         return batch

# def compute_metrics(eval_predictions):
#     predictions, label_ids = eval_predictions
#     preds = np.argmax(predictions, axis=1)
    
#     map3_score = 0.0
#     for pred_scores, true_label in zip(predictions, label_ids):
#         top_3 = np.argsort(pred_scores)[::-1][:3]
#         if true_label in top_3:
#             position = np.where(top_3 == true_label)[0][0] + 1
#             map3_score += 1.0 / position
            
#     return {
#         "accuracy": (preds == label_ids).astype(np.float32).mean().item(),
#         "map@3": map3_score / len(label_ids)
#     }

# def run_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame, model_name: str = "distilroberta-base") -> pd.DataFrame:
#     options = ['A', 'B', 'C', 'D', 'E']
#     option_to_index = {opt: i for i, opt in enumerate(options)}
#     index_to_option = {i: opt for i, opt in enumerate(options)}

#     train_df = train_df.copy()
#     test_df = test_df.copy()

#     train_df['label'] = train_df['answer'].map(option_to_index)
#     test_df['label'] = 0 

#     knowledge_base = train_df.apply(
#         lambda row: f"Question: {row['prompt']} Answer: {row[row['answer']]}", axis=1
#     ).tolist()

#     retriever = TFIDFRetriever(knowledge_base)
#     train_df['context'] = retriever.retrieve(train_df['prompt'].tolist(), top_k=3)
#     test_df['context'] = retriever.retrieve(test_df['prompt'].tolist(), top_k=3)

#     train_df, eval_df = train_test_split(
#         train_df, 
#         test_size=0.1, 
#         stratify=train_df['label'], 
#         random_state=42
#     )

#     train_ds = Dataset.from_pandas(train_df)
#     eval_ds = Dataset.from_pandas(eval_df)
#     test_ds = Dataset.from_pandas(test_df)

#     tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

#     def preprocess_function(examples):
#         first_sentences = []
#         for ctx, prompt in zip(examples["context"], examples["prompt"]):
#             if str(ctx).strip():
#                 combined = f"Context: {ctx}\nQuestion: {prompt}"
#             else:
#                 combined = str(prompt)
#             first_sentences.append([combined] * 5)
            
#         second_sentences = [[str(examples[opt][i]) for opt in options] for i in range(len(examples["prompt"]))]
        
#         first_sentences = sum(first_sentences, [])
#         second_sentences = sum(second_sentences, [])
        
#         tokenized_examples = tokenizer(
#             first_sentences,
#             second_sentences,
#             truncation="only_first",  
#             max_length=256, 
#             padding=False
#         )
#         return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

#     tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=[c for c in train_ds.column_names if c != 'label'])
#     tokenized_eval = eval_ds.map(preprocess_function, batched=True, remove_columns=[c for c in eval_ds.column_names if c != 'label'])
#     tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=[c for c in test_ds.column_names if c != 'label'])

#     model = AutoModelForMultipleChoice.from_pretrained(model_name)
#     use_fp16 = torch.cuda.is_available() and ('roberta' in model_name.lower())

#     training_args = TrainingArguments(
#         output_dir="./results",
#         eval_strategy="epoch",             
#         save_strategy="epoch",             
#         logging_strategy="epoch",          
#         load_best_model_at_end=True,       
#         metric_for_best_model="map@3",
#         greater_is_better=True,
#         learning_rate=6e-6,                # Baseline LR
#         per_device_train_batch_size=1,     # Baseline Batch
#         per_device_eval_batch_size=2,
#         gradient_accumulation_steps=2,     # Baseline Accumulation
#         max_grad_norm=1.0,                 # Explicit gradient clipping
#         warmup_ratio=0.2,                  # Baseline Warmup
#         lr_scheduler_type="linear",        
#         num_train_epochs=4,                # Baseline Epochs
#         weight_decay=0.01,
#         fp16=use_fp16,
#         report_to="none"
#     )

#     trainer = CustomTrainer(
#         model=model,
#         args=training_args,
#         train_dataset=tokenized_train,
#         eval_dataset=tokenized_eval,
#         processing_class=tokenizer,
#         data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
#         compute_metrics=compute_metrics,
#         callbacks=[MonitorCallback()]
#     )

#     trainer.train()

#     predictions = trainer.predict(tokenized_test).predictions
#     top_3_indices = np.argsort(predictions, axis=1)[:, ::-1][:, :3]
    
#     top_3_predictions = []
#     for indices in top_3_indices:
#         pred_str = " ".join([index_to_option[idx] for idx in indices])
#         top_3_predictions.append(pred_str)

#     submission_df = pd.DataFrame({
#         'id': test_df['id'],
#         'Prediction': top_3_predictions
#     })

#     return submission_df

# if __name__ == "__main__":
#     run_pipeline(train_df,test_df)

In [7]:
# sub_df = run_scratch_pipeline(train_df,test_df,'/kaggle/working/model')

In [8]:
# sub_df.to_csv('submission.csv',index=False)

In [9]:
# sub_df

In [10]:
# import pandas as pd
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split 
# from collections import Counter
# import re
# import math
# import os
# import pickle 
# from gensim.models import FastText 
# from monitor.wandb_monitor import TrainMonitor



# class CustomFastTextTokenizer:
#     def __init__(self):
#         self.word2idx = {'<PAD>': 0, '<UNK>': 1}
#         self.idx2word = {0: '<PAD>', 1: '<UNK>'}
#         self.vocab_size = 2
        
#     def clean_text(self, text):
#         text = str(text).lower()
#         text = re.sub(r'[^a-z0-9\s]', ' ', text)
#         return text.split()

#     def train_and_build_matrix(self, texts, d_model=128):
#         print("Tokenizing corpus for FastText...")
#         sentences = [self.clean_text(text) for text in texts]
        
#         print(f"Training custom FastText model on {len(sentences)} sequences...")
#         ft_model = FastText(sentences=sentences, vector_size=d_model, window=5, min_count=1, workers=4, epochs=15)
#         words = list(ft_model.wv.index_to_key)
#         self.vocab_size = len(words) + 2
#         embedding_matrix = np.zeros((self.vocab_size, d_model))
#         embedding_matrix[1] = np.random.normal(scale=0.1, size=(d_model,))
        
#         print("Transferring weights to PyTorch embedding matrix...")
#         for i, word in enumerate(words):
#             idx = i + 2 # Shift by 2 because 0=PAD, 1=UNK
#             self.word2idx[word] = idx
#             self.idx2word[idx] = word
#             embedding_matrix[idx] = ft_model.wv[word]
            
#         print(f"Custom FastText Vocabulary built with {self.vocab_size} tokens.")
#         return torch.tensor(embedding_matrix, dtype=torch.float32)

#     def encode(self, text, max_len):
#         words = self.clean_text(text)
#         tokens = [self.word2idx.get(w, 1) for w in words] # 1 is <UNK>
        
#         # Truncate or Pad
#         if len(tokens) > max_len:
#             tokens = tokens[:max_len]
#         else:
#             tokens = tokens + [0] * (max_len - len(tokens)) # 0 is <PAD>
#         return tokens

# class ScratchMCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_len=128, is_test=False):
#         self.df = df
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#         self.is_test = is_test
#         self.options = ['A', 'B', 'C', 'D', 'E']

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         prompt = str(row['prompt'])
        
#         # Encode [Prompt + Option] for all 5 choices
#         input_ids = []
#         for opt in self.options:
#             combined_text = prompt + " " + str(row[opt])
#             encoded = self.tokenizer.encode(combined_text, self.max_len)
#             input_ids.append(encoded)
            
#         item = {
#             'input_ids': torch.tensor(input_ids, dtype=torch.long)
#         }
        
#         if not self.is_test:
#             ans_idx = self.options.index(row['answer'])
#             item['label'] = torch.tensor(ans_idx, dtype=torch.long)
            
#         return item

# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model, dropout=0.1, max_len=500):
#         super().__init__()
#         self.dropout = nn.Dropout(p=dropout)
#         position = torch.arange(max_len).unsqueeze(1)
#         div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
#         pe = torch.zeros(max_len, 1, d_model)
#         pe[:, 0, 0::2] = torch.sin(position * div_term)
#         pe[:, 0, 1::2] = torch.cos(position * div_term)
#         self.register_buffer('pe', pe)

#     def forward(self, x):
#         x = x + self.pe[:x.size(0)]
#         return self.dropout(x)

# class TinyMCQModel(nn.Module):
#     def __init__(self, vocab_size, d_model=300, nhead=6, num_layers=1, dropout=0.3, pretrained_embeddings=None):
#         super().__init__()
#         if pretrained_embeddings is not None:
#             self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False, padding_idx=0)
#         else:
#             self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
            
#         self.pos_encoder = PositionalEncoding(d_model, dropout)
        
#         encoder_layers = nn.TransformerEncoderLayer(
#             d_model=d_model, 
#             nhead=nhead, 
#             dim_feedforward=512, 
#             dropout=dropout,
#             batch_first=True
#         )
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
#         self.classifier = nn.Sequential(
#             nn.Linear(d_model, 64),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(64, 1)
#         )

#     def forward(self, input_ids):
#         batch_size, num_opts, seq_len = input_ids.shape
#         x = input_ids.view(batch_size * num_opts, seq_len)
#         x = self.embedding(x) 
#         x = self.transformer_encoder(x) 
#         x = x.mean(dim=1)
#         logits = self.classifier(x) 
#         logits = logits.view(batch_size, num_opts)
#         return logits

# # Training

# def run_scratch_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame, save_dir: str = None):
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(f"Using device: {device}")
    
#     monitor = TrainMonitor(
#     project="24f1000781-t22026",
#     model_name="fasttext_MLP",
#     version="v1",
#     experiment="Baseline",
#     config={
#         "lr":2e-5,
#         "batch_size":16,
#         "epochs":8,
#         "max length":512
#         }
#     )
    
#     train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
#     train_df = train_df.reset_index(drop=True)
#     val_df = val_df.reset_index(drop=True)

#     tokenizer = CustomFastTextTokenizer()
#     all_text = train_df['prompt'].tolist() + val_df['prompt'].tolist() + test_df['prompt'].tolist()
#     for opt in ['A', 'B', 'C', 'D', 'E']:
#         all_text.extend(train_df[opt].tolist())
#         all_text.extend(val_df[opt].tolist())
#         all_text.extend(test_df[opt].tolist())

#     D_MODEL = 256 
#     pretrained_embeddings = tokenizer.train_and_build_matrix(all_text, d_model=D_MODEL)

#     max_len = 256
#     train_ds = ScratchMCQDataset(train_df, tokenizer, max_len=max_len)
#     val_ds = ScratchMCQDataset(val_df, tokenizer, max_len=max_len) 
#     test_ds = ScratchMCQDataset(test_df, tokenizer, max_len=max_len, is_test=True)

#     train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
#     val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
#     test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
#     model = TinyMCQModel(
#         vocab_size=tokenizer.vocab_size, 
#         d_model=D_MODEL,
#         nhead=4,        
#         num_layers=2, 
#         dropout=0.3,
#         pretrained_embeddings=pretrained_embeddings
#     ).to(device)
    
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

#     epochs = 15
#     print(f"\nStarting Training for {epochs} Epochs...")
#     for epoch in range(epochs):
#         model.train()
#         total_train_loss = 0
#         train_correct = 0
#         train_total = 0
        
#         for batch in train_loader:
#             inputs = batch['input_ids'].to(device)
#             labels = batch['label'].to(device)
            
#             optimizer.zero_grad()
#             logits = model(inputs) 
            
#             loss = criterion(logits, labels)
#             loss.backward()
#             optimizer.step()
            
#             total_train_loss += loss.item()
#             preds = torch.argmax(logits, dim=1)
#             train_correct += (preds == labels).sum().item()
#             train_total += labels.size(0)
            
#         train_loss = total_train_loss / len(train_loader)
#         train_acc = train_correct / train_total

#         # Validate
#         model.eval()
#         total_val_loss = 0
#         val_correct = 0
#         val_total = 0
        
#         with torch.no_grad():
#             for batch in val_loader:
#                 inputs = batch['input_ids'].to(device)
#                 labels = batch['label'].to(device)
                
#                 logits = model(inputs)
#                 loss = criterion(logits, labels)
                
#                 total_val_loss += loss.item()
#                 preds = torch.argmax(logits, dim=1)
#                 val_correct += (preds == labels).sum().item()
#                 val_total += labels.size(0)

#         val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
#         val_acc = val_correct / val_total if val_total > 0 else 0.0

#         try:
#             monitor.monitor({
#                 "epoch": epoch,
#                 "train/loss": train_loss,
#                 "train/accuracy": train_acc,
#                 "validation/loss": val_loss,
#                 "validation/accuracy": val_acc,
#             })
#         except NameError:
#             pass 

#         print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

#     if save_dir:
#         print(f"\nSaving model and tokenizer to {save_dir}...")
#         os.makedirs(save_dir, exist_ok=True)
#         # Save PyTorch weights
#         torch.save(model.state_dict(), os.path.join(save_dir, "tiny_mcq_model.pth"))
#         # Save Tokenizer
#         with open(os.path.join(save_dir, "tokenizer.pkl"), "wb") as f:
#             pickle.dump(tokenizer, f)
#         print("Save complete!")

# # Results

#     print("\nGenerating Test Predictions...")
#     model.eval()
#     all_preds = []
#     options = ['A', 'B', 'C', 'D', 'E']
    
#     with torch.no_grad():
#         for batch in test_loader:
#             inputs = batch['input_ids'].to(device)
#             logits = model(inputs)
            
#             # Sort to get top 3 indices
#             scores = logits.cpu().numpy()
#             top_3_idx = np.argsort(scores, axis=1)[:, ::-1][:, :3]
            
#             for row in top_3_idx:
#                 pred_str = " ".join([options[i] for i in row])
#                 all_preds.append(pred_str)

#     submission_df = pd.DataFrame({
#         'id': test_df['id'],
#         'Prediction': all_preds
#     })
    
#     return submission_df